# Modul 6: Supervised Learning - Feature Engineering & Penanganan Data Imbalance
**Project-Based Internship VINIX7**

Kelompok 3 (Universitas Sultan Ageng Tirtayasa)

Anggota Kelompok: Ahmad Jumhadi, Azhriler Lintang, Aura Salsa Azzahra

## Instalasi & Import Library

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

!pip install xgboost
!pip install imbalanced-learn

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

print("Library berhasil diimport.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.4/235.4 kB 10.7 MB/s eta 0:00:00
Library berhasil diimport.


Kode ini merupakan tahap persiapan dalam proses analisis data, yaitu semua library yang dibutuhkan di-import terlebih dahulu. Library pandas dan numpy digunakan untuk mengelola dan memanipulasi data. Sementara itu, warnings.filterwarnings('ignore') dipakai agar tampilan output di notebook lebih rapi dan tidak terlalu dipenuhi pesan warning yang tidak terlalu krusial.

Selanjutnya, dilakukan instalasi xgboost dan imbalanced-learn. XGBoost akan digunakan sebagai algoritma utama untuk klasifikasi, sedangkan imbalanced-learn khususnya SMOTE digunakan untuk mengatasi masalah ketidakseimbangan data.

Sementara itu, bagian import dari sklearn memperlihatkan bahwa pipeline yang dibangun cukup lengkap. Terdapat proses pembagian data train dan test, preprocessing untuk berbagai tipe data seperti teks menggunakan TF-IDF, data numerik dengan scaling, dan data kategorikal dengan encoding. Semua ini kemudian akan digabungkan menggunakan ColumnTransformer dalam satu pipeline agar lebih terstruktur. Selain itu, digunakan juga classification_report untuk evaluasi performa model.

---
## Tahap 1: Data Cleansing & Handling 'Neutral'

In [ ]:
df = pd.read_csv('/content//tokopedia_product_reviews_2025.csv')

print("Shape awal:", df.shape)
print("\nDistribusi sentiment_label:")
print(df['sentiment_label'].value_counts())

Shape awal: (65543, 13)

Distribusi sentiment_label:
sentiment_label
positive    63943
neutral       802
negative      798
Name: count, dtype: int64


Pada tahap awal ini, data di-load menggunakan pandas dan langsung dilihat bentuk serta distribusi labelnya. Hasilnya menunjukkan bahwa terdapat total 65543 data, dengan distribusi yang sangat tidak seimbang, karena hampir semua review masuk ke kategori positive sebesar 63943 data, sementara neutral berjumlah sebesar 802 data dan negative jumlahnya 798 data.

In [ ]:
df = df[df['sentiment_label'] != 'neutral'].copy()

print("Shape setelah hapus neutral:", df.shape)
print("\nDistribusi setelah cleansing:")
print(df['sentiment_label'].value_counts())

Shape setelah hapus neutral: (64741, 13)

Distribusi setelah cleansing:
sentiment_label
positive    63943
negative      798
Name: count, dtype: int64


Selanjutnya dilakukan proses cleansing dengan menghapus data yang berlabel neutral. Setelah dihapus, jumlah data berkurang menjadi 64741 data, dan sekarang hanya tersisa dua label yaitu positive dan negative. Hal ini dilakukan agar lebih fokus membedakan antara komplain (negative) dan bukan komplain (positive). Karena dari sisi bisnis, tujuan utama klien adalah mendeteksi komplain secepat mungkin, jadi label neutral dianggap kurang memberikan nilai yang jelas.

In [ ]:
drop_cols = ['review_date', 'review_id', 'product_name', 'product_url',
             'product_id', 'shop_id', 'rating']

df = df.drop(columns=drop_cols)

print("Kolom setelah drop:", df.columns.tolist())
print("Shape akhir Tahap 1:", df.shape)
df.head(3)

Kolom setelah drop: ['review_text', 'product_category', 'product_variant', 'product_price', 'sold_count', 'sentiment_label']
Shape akhir Tahap 1: (64741, 6)


,review_text,product_category,product_variant,product_price,sold_count,sentiment_label
0,baru sekali ini terima brg dr belanja online d...,Makanan & Minuman,Box Polos,87000,1000000,positive
1,cocok bgt aku sama telur nya. nga Amis menurut...,Makanan & Minuman,Box Polos,87000,1000000,positive
2,Telornya sudah sampai di rumah dengan kemasan ...,Makanan & Minuman,Box Polos,87000,1000000,positive


Setelah itu, dilakukan penghapusan beberapa kolom yang dianggap tidak relevan atau berpotensi menyebabkan data leakage, yaitu review_date, review_id, product_name, product_url, product_id, shop_id,
dan rating. Setelah kolom-kolom ini dihapus, tersisa 6 kolom utama yang lebih relevan, yaitu review_text, product_category, product_variant, product_price, sold_count, dan sentiment_label.

---
## Tahap 2: Feature Engineering & Preprocessing

In [ ]:
df['review_length'] = df['review_text'].apply(len)

print("Statistik review_length:")
print(df['review_length'].describe())

Statistik review_length:
count    64741.000000
mean        78.165954
std        174.637180
min          4.000000
25%         36.000000
50%         59.000000
75%         95.000000
max      32857.000000
Name: review_length, dtype: float64


Pada tahap ini dilakukan proses feature engineering, yaitu menambahkan fitur baru bernama review_length yang berisi jumlah karakter teks dari kolom review_text. Fitur review_length ditambahkan karena panjang teks sering kali berkorelasi dengan tingkat emosi atau detail keluhan pelanggan. Review negatif cenderung lebih panjang karena berisi penjelasan masalah, sehingga fitur ini berpotensi membantu model membedakan pola antara review positif dan negatif. Dari hasil statistiknya, terlihat bahwa rata-rata panjang review sekitar 78 karakter, tetapi variasinya cukup besar sampai puluhan ribu karakter.

In [ ]:
df['sentiment_label'] = df['sentiment_label'].map({'positive': 1, 'negative': 0})

X = df.drop(columns=['sentiment_label'])
y = df['sentiment_label']

print("Shape X:", X.shape)
print("Distribusi y:")
print(y.value_counts())
print(f"\nRasio imbalance: {y.value_counts()[1]}/{y.value_counts()[0]} = {y.value_counts()[1]/y.value_counts()[0]:.1f}x")

Shape X: (64741, 6)
Distribusi y:
sentiment_label
1    63943
0      798
Name: count, dtype: int64

Rasio imbalance: 63943/798 = 80.1x


Selanjutnya, label sentimen diubah jadi bentuk numerik, yaitu positive = 1 dan negative = 0. Setelah itu dipisahkan antara fitur (X) dan target (y). Dari distribusi target, terlihat bahwa datanya sangat tidak seimbang, dengan rasio imbalance sekitar 80:1.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Shape X_train:", X_train.shape)
print("Shape X_test: ", X_test.shape)
print("\nDistribusi y_train:")
print(y_train.value_counts())
print("\nDistribusi y_test:")
print(y_test.value_counts())

Shape X_train: (51792, 6)
Shape X_test:  (12949, 6)

Distribusi y_train:
sentiment_label
1    51154
0      638
Name: count, dtype: int64

Distribusi y_test:
sentiment_label
1    12789
0      160
Name: count, dtype: int64


Sebelum masuk tahap transformasi, untuk menghindari data leakage, dilakukan proses split data dengan dataset dibagi menjadi 80% training dan 20% testing. Hasilnya menunjukkan bahwa distribusi pada train dan test tetap mencerminkan kondisi awal yang imbalanced.

In [ ]:
text_col       = 'review_text'
cat_cols       = ['product_category', 'product_variant']
num_cols       = ['product_price', 'sold_count', 'review_length']

preprocessor = ColumnTransformer(
    transformers=[
        ('tfidf', TfidfVectorizer(max_features=1000), text_col),
        ('ohe',   OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('scaler', StandardScaler(), num_cols)
    ],
    remainder='drop'
)


X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print("Shape X_train_processed:", X_train_processed.shape)
print("Shape X_test_processed: ", X_test_processed.shape)

Shape X_train_processed: (51792, 5972)
Shape X_test_processed:  (12949, 5972)


Selanjutnya dilakukan preprocessing menggunakan ColumnTransformer yang menggabungkan beberapa teknik sesuai tipe datanya. Kolom teks diubah jadi representasi numerik menggunakan TF-IDF dengan batas 1000 fitur. Kolom kategorikal seperti product_category dan product_variant diubah dengan One-Hot Encoding supaya bisa dipahami model. Sementara itu, fitur numerik seperti product_price, sold_count, dan review_length dinormalisasi dengan StandardScaler agar skalanya seragam. Proses fit dan transform dilakukan di data train, lalu data test hanya di-transform. Hasilnya menunjukkan bahwa jumlah fitur meningkat cukup drastis menjadi sekitar 5972 fitur.

---
## Tahap 3: Eksperimen Model (Baseline)

In [ ]:
baseline_model = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

baseline_model.fit(X_train_processed, y_train)
print("Model baseline selesai dilatih.")

Model baseline selesai dilatih.


Pada tahap ini, model baseline dilatih menggunakan algoritma XGBoost dengan parameter default. Di tahap ini, belum ada perlakuan khusus terhadap data imbalance, jadi model belajar dari distribusi data apa adanya. Selanjutnya, model yang sudah dilatih ini digunakan untuk melakukan prediksi pada data test (X_test_processed).

In [ ]:
y_pred_baseline = baseline_model.predict(X_test_processed)

print("=== Classification Report — Baseline Model ===")
print(classification_report(
    y_test,
    y_pred_baseline,
    target_names=['Negative (0)', 'Positive (1)']
))

=== Classification Report — Baseline Model ===
              precision    recall  f1-score   support

Negative (0)       0.67      0.21      0.32       160
Positive (1)       0.99      1.00      0.99     12789

    accuracy                           0.99     12949
   macro avg       0.83      0.61      0.66     12949
weighted avg       0.99      0.99      0.99     12949



**Interpretasi 1:** Dari hasil evaluasi, terlihat bahwa performa model untuk kelas negatif cukup rendah, terutama pada nilai recall yang hanya sebesar 0.21. Artinya, dari total 160 data komplain, model hanya berhasil mendeteksi sekitar 21% saja, sementara sisanya tidak teridentifikasi sebagai komplain. Sebaliknya, untuk kelas positif, model memiliki performa yang hampir sempurna dengan recall 1.00. Hal ini membuat akurasi keseluruhan terlihat sangat tinggi sebesar 99%, padahal sebenarnya model tidak seimbang dalam mengenali kedua kelas.
Secara teknis, kondisi ini terjadi karena data yang digunakan sangat imbalanced, di mana jumlah data positive jauh lebih banyak dibanding negative. Akibatnya, model cenderung “bermain aman” dengan lebih sering memprediksi ke kelas positive karena itu yang paling sering muncul di data training. Tanpa adanya penanganan khusus seperti balancing atau penyesuaian bobot kelas, model tidak cukup “dipaksa” untuk belajar pola dari data negative yang jumlahnya sedikit.
Dari sisi bisnis, ini cukup berisiko jika model baseline langsung di-deploy. Walaupun akurasi tinggi, model justru gagal mendeteksi sebagian besar komplain pelanggan. Dampaknya, banyak keluhan yang tidak tertangani oleh tim Customer Service, sehingga bisa meningkatkan ketidakpuasan pelanggan dan berpotensi menyebabkan mereka beralih ke kompetitor.

---
## Tahap 4: Eksperimen Penanganan Imbalance & Tuning

In [ ]:
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
spw = pos_count / neg_count

print(f"Kelas 0 (negatif) di train: {neg_count}")
print(f"Kelas 1 (positif) di train: {pos_count}")
print(f"scale_pos_weight (sebagai referensi): {spw:.2f}")

Kelas 0 (negatif) di train: 638
Kelas 1 (positif) di train: 51154
scale_pos_weight (sebagai referensi): 80.18


Pada tahap ini, dilakukan perhitungan jumlah masing-masing kelas pada data training. Terlihat bahwa kelas negative hanya berjumlah 638, sedangkan kelas positive mencapai 51154. Dari sini dihitung juga nilai scale_pos_weight sebesar 80.18, yang sebenarnya bisa digunakan sebagai referensi untuk menangani imbalance langsung di model XGBoost.

In [ ]:
imb_pipeline = ImbPipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ('xgb',   XGBClassifier(
                  random_state=42,
                  eval_metric='logloss',
                  use_label_encoder=False
              ))
])

param_grid = {
    'xgb__n_estimators':    [100, 200],
    'xgb__max_depth':       [3, 5],
    'xgb__scale_pos_weight': [1, spw]
}

print("Pipeline dan param_grid siap.")

Pipeline dan param_grid siap.


Selanjutnya, pipeline dibangun menggunakan ImbPipeline dari imblearn. Pipeline ini menggabungkan dua proses sekaligus, yaitu SMOTE untuk oversampling data minoritas, lalu dilanjutkan dengan model XGBoost. Selain itu, disiapkan juga param_grid untuk mencoba beberapa kombinasi parameter seperti jumlah estimator dan kedalaman tree, serta disiapkan juga opsi scale_pos_weight.

In [ ]:
imb_pipeline = ImbPipeline(steps=[
    ('smote', SMOTE(random_state=42, sampling_strategy=0.3)),
    ('xgb',   XGBClassifier(
                  random_state=42,
                  eval_metric='logloss',
                  tree_method='hist',
                  n_jobs=1
              ))
])

param_grid = {
    'xgb__n_estimators': [100, 200],
    'xgb__max_depth':    [3, 5],
}

grid_search = GridSearchCV(
    estimator=imb_pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='recall_macro',
    n_jobs=1,
    verbose=1
)

grid_search.fit(X_train_processed, y_train)

print("\nParameter terbaik:")
print(grid_search.best_params_)
print(f"Best CV Score (recall_macro): {grid_search.best_score_:.4f}")

Fitting 3 folds for each of 4 candidates, totalling 12 fits

Parameter terbaik:
{'xgb__max_depth': 3, 'xgb__n_estimators': 200}
Best CV Score (recall_macro): 0.6643


Selanjutnya, pipeline diperbarui dengan pengaturan SMOTE yang lebih terkontrol menggunakan sampling_strategy=0.3. Artinya, jumlah data minoritas tidak disamakan sepenuhnya dengan mayoritas, tapi hanya ditingkatkan sampai 30% dari jumlah kelas mayoritas. GridSearchCV kemudian dijalankan dengan 3-fold cross-validation dan menggunakan metrik recall_macro, yang berarti fokus evaluasi ada pada kemampuan model mengenali kedua kelas secara seimbang, terutama kelas negatif. Hasil dari GridSearch menunjukkan parameter terbaik adalah max_depth=3 dan n_estimators=200, dengan nilai recall_macro 0.6643.

In [ ]:
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test_processed)

print("=== Classification Report — Tuned Model (SMOTE + XGBoost) ===")
print(classification_report(
    y_test,
    y_pred_tuned,
    target_names=['Negative (0)', 'Positive (1)']
))

=== Classification Report — Tuned Model (SMOTE + XGBoost) ===
              precision    recall  f1-score   support

Negative (0)       0.47      0.38      0.42       160
Positive (1)       0.99      0.99      0.99     12789

    accuracy                           0.99     12949
   macro avg       0.73      0.69      0.71     12949
weighted avg       0.99      0.99      0.99     12949



**Interpretasi 2:** Pendekatan yang kami gunakan adalah kombinasi oversampling menggunakan SMOTE dan tuning model XGBoost. SMOTE bekerja dengan membuat data sintetis untuk kelas minoritas, sehingga distribusi data menjadi lebih seimbang dan model punya lebih banyak contoh untuk belajar pola komplain.

Dari hasil evaluasi, terlihat ada peningkatan recall pada kelas negative menjadi 0.38, tapi di sisi lain precision untuk kelas negative justru tidak terlalu tinggi, menjadi 0.47. Ini menunjukkan adanya trade-off, yaitu model jadi lebih sering mendeteksi komplain (recall naik), tapi juga jadi lebih sering salah menandai review positif sebagai negatif (precision tidak terlalu tinggi). Secara teknis, ini wajar karena setelah data diseimbangkan, model jadi lebih sensitif terhadap pola minoritas, sehingga batas keputusannya bergeser dan lebih berani mengklasifikasikan sesuatu sebagai negatif. Selain itu, penggunaan recall_macro sebagai scoring di GridSearch juga mendorong model untuk tidak mengabaikan kelas minoritas.

In [ ]:
from sklearn.metrics import recall_score, precision_score, f1_score

metrics = {
    'Model'        : ['Baseline', 'Tuned (SMOTE+XGB)'],
    'Recall Neg(0)': [
        recall_score(y_test, y_pred_baseline, pos_label=0),
        recall_score(y_test, y_pred_tuned,    pos_label=0)
    ],
    'Precision Neg(0)': [
        precision_score(y_test, y_pred_baseline, pos_label=0),
        precision_score(y_test, y_pred_tuned,    pos_label=0)
    ],
    'F1 Neg(0)'    : [
        f1_score(y_test, y_pred_baseline, pos_label=0),
        f1_score(y_test, y_pred_tuned,    pos_label=0)
    ],
    'Recall Pos(1)': [
        recall_score(y_test, y_pred_baseline, pos_label=1),
        recall_score(y_test, y_pred_tuned,    pos_label=1)
    ],
}

comparison_df = pd.DataFrame(metrics)
print(comparison_df.to_string(index=False))

            Model  Recall Neg(0)  Precision Neg(0)  F1 Neg(0)  Recall Pos(1)
         Baseline        0.21250          0.666667   0.322275       0.998671
Tuned (SMOTE+XGB)        0.38125          0.469231   0.420690       0.994605


---
## Tahap 5: Kesimpulan Akhir & Keputusan Bisnis

In [ ]:
from sklearn.metrics import confusion_matrix

cm_baseline = confusion_matrix(y_test, y_pred_baseline, labels=[0,1])
cm_tuned    = confusion_matrix(y_test, y_pred_tuned,    labels=[0,1])

def parse_cm(cm, label):
    tn, fn, fp, tp = cm.ravel()
    print(f"\n--- {label} ---")
    print(f"  True Negative  (TN) = {tn}  → Komplain BENAR diidentifikasi sebagai komplain")
    print(f"  False Positive (FP) = {fp}  → Review positif SALAH diklasifikasi sebagai komplain")
    print(f"  False Negative (FN) = {fn}  → Komplain TERLEWAT (tidak terdeteksi oleh model)")
    print(f"  True Positive  (TP) = {tp}  → Review positif BENAR diidentifikasi")

    if (tn + fn) > 0:
        recall_neg = tn / (tn + fn)
        print(f"  Recall Kelas 0      = {recall_neg:.4f}")

    if (tn + fn) > 0:
        print(f"  Komplain terlewat: {fn} dari {tn+fn} total komplain ({fn/(tn+fn)*100:.1f}%)")

parse_cm(cm_baseline, "Baseline Model")
parse_cm(cm_tuned,    "Tuned Model (SMOTE+XGB)")


--- Baseline Model ---
  True Negative  (TN) = 34  → Komplain BENAR diidentifikasi sebagai komplain
  False Positive (FP) = 17  → Review positif SALAH diklasifikasi sebagai komplain
  False Negative (FN) = 126  → Komplain TERLEWAT (tidak terdeteksi oleh model)
  True Positive  (TP) = 12772  → Review positif BENAR diidentifikasi
  Recall Kelas 0      = 0.2125
  Komplain terlewat: 126 dari 160 total komplain (78.8%)

--- Tuned Model (SMOTE+XGB) ---
  True Negative  (TN) = 61  → Komplain BENAR diidentifikasi sebagai komplain
  False Positive (FP) = 69  → Review positif SALAH diklasifikasi sebagai komplain
  False Negative (FN) = 99  → Komplain TERLEWAT (tidak terdeteksi oleh model)
  True Positive  (TP) = 12720  → Review positif BENAR diidentifikasi
  Recall Kelas 0      = 0.3812
  Komplain terlewat: 99 dari 160 total komplain (61.9%)


**Interpretasi 3:** Berdasarkan hasil evaluasi confusion matrix dan classification report, model baseline menunjukkan performa yang kurang baik dalam mendeteksi komplain pelanggan. Dari total 160 komplain, sebanyak 126 kasus (78.8%) tidak berhasil terdeteksi oleh model. Hal ini terlihat dari nilai recall kelas negatif yang hanya sebesar 0.21, yang menunjukkan bahwa model sangat bias terhadap kelas mayoritas (positive). Setelah dilakukan penanganan imbalance menggunakan SMOTE dan tuning model, terjadi peningkatan kemampuan model dalam mengenali komplain. Jumlah komplain yang berhasil dideteksi meningkat dari 34 menjadi 61 kasus, dan jumlah komplain yang terlewat menurun dari 126 menjadi 99 kasus (61.9%). Hal ini tercermin dari peningkatan recall kelas negatif menjadi 0.38.

Namun, peningkatan ini disertai dengan trade-off, yaitu meningkatnya jumlah false alarm (False Positive), dari 17 menjadi 69 kasus. Artinya, lebih banyak review positif yang salah diklasifikasikan sebagai komplain, sehingga tim Customer Service perlu melakukan verifikasi tambahan. Dari perspektif bisnis, trade-off ini masih dapat diterima. Dampak dari False Positive hanya berupa tambahan beban kerja operasional, sedangkan False Negative (komplain yang terlewat) berisiko langsung terhadap kepuasan pelanggan dan potensi churn. Oleh karena itu, model hasil tuning lebih layak digunakan sebagai sistem triase untuk memprioritaskan tiket Customer Service dibandingkan model baseline, karena lebih mampu mengurangi jumlah komplain yang tidak tertangani.